# bashkit in Jupyter

bashkit gives you a safe, sandboxed bash environment that integrates naturally
with Jupyter's async model.

- **`await bash.execute()`** — the idiomatic Jupyter form; callbacks run on the notebook's own event loop
- **`bash.execute_sync()`** — also works in Jupyter even though a loop is already running
- **`custom_builtins`** — Python functions (sync *or* async) registered as bash commands
- **ContextVars** — values set in a cell propagate automatically into callbacks

```
pip install bashkit
```

In [ ]:
import asyncio
import contextvars

from bashkit import Bash, BuiltinContext, ScriptedTool

print(f"Loop running: {asyncio.get_event_loop().is_running()}")

## Async execution — the natural Jupyter way

Jupyter supports top-level `await`, so `await bash.execute()` is the idiomatic
form. Async `custom_builtins` callbacks run on the notebook's own event loop.

In [ ]:
bash = Bash()
result = await bash.execute("""
    echo 'files created:'
    for f in report data config; do
        touch /workspace/$f.txt
        echo "  /workspace/$f.txt"
    done
""")
print(result.stdout)

## Sync execution also works

`execute_sync()` works even though Jupyter's event loop is already running.
Internally it dispatches async callbacks to a background thread when needed,
so there is no conflict.

In [ ]:
result = bash.execute_sync("ls /workspace/")
print(result.stdout)

## Persistent workspace across cells

The virtual filesystem and shell variables survive across calls on the same
`Bash` instance — just like a real shell session.

In [ ]:
bash.execute_sync("""
    printf 'name,score\nalice,95\nbob,87\ncarol,92\ndave,78\n' > /workspace/scores.csv
    export THRESHOLD=90
""")
print("Data written.")

In [ ]:
# Variables and files from the previous cell are still there
result = bash.execute_sync("awk -F, -v t=$THRESHOLD '$2 >= t' /workspace/scores.csv")
print("High scorers (>= $THRESHOLD):")
print(result.stdout)

## Custom Python commands — sync

Register any Python function as a bash command via `custom_builtins=`.
The callback receives a `BuiltinContext` with `argv`, `stdin`, `env`, and `cwd`.

In [ ]:
import json

def summarize(ctx: BuiltinContext) -> str:
    """Read CSV from stdin, return JSON summary."""
    lines = [l for l in (ctx.stdin or "").strip().splitlines() if "," in l]
    header, *rows = lines
    cols = header.split(",")
    records = [dict(zip(cols, r.split(","))) for r in rows]
    values = [float(r[cols[1]]) for r in records]
    return json.dumps({
        "count": len(values),
        "avg": round(sum(values) / len(values), 1),
        "max": max(values),
        "min": min(values),
    }) + "\n"

bash = Bash(custom_builtins={"summarize": summarize})
bash.execute_sync("printf 'name,score\nalice,95\nbob,87\ncarol,92\ndave,78\n' > /data/scores.csv")

result = bash.execute_sync("cat /data/scores.csv | summarize | jq .")
print(result.stdout)

## Custom Python commands — async

Callbacks can be `async def` — bashkit drives them to completion whether you
call `execute_sync()` or `await execute()`.  This is useful for async I/O
like HTTP requests or database queries inside a bash pipeline.

In [ ]:

# Async callback used via execute_sync() — works fine in Jupyter
async def enrich(ctx: BuiltinContext) -> str:
    name = ctx.argv[0] if ctx.argv else "unknown"
    await asyncio.sleep(0)  # replace with real async I/O
    metadata = {"name": name, "team": "engineering", "active": True}
    return json.dumps(metadata) + "\n"

bash = Bash(custom_builtins={"enrich": enrich})

result = bash.execute_sync("enrich alice | jq -r '.name'")
print(result.stdout)


In [ ]:
# Same callback, this time via await execute() — callback runs on Jupyter's loop
caller_loop = asyncio.get_running_loop()
seen_loops: list = []

async def enrich_and_record(ctx: BuiltinContext) -> str:
    seen_loops.append(asyncio.get_running_loop() is caller_loop)
    await asyncio.sleep(0)
    name = ctx.argv[0] if ctx.argv else "?"
    return json.dumps({"name": name, "enriched": True}) + "\n"

bash = Bash(custom_builtins={"enrich": enrich_and_record})
result = await bash.execute("enrich bob | jq -r '.name'")

print(result.stdout.strip())
print(f"Callback ran on Jupyter's loop: {seen_loops[0]}")

## ContextVar propagation

Values stored in `contextvars.ContextVar` before calling `execute()` or
`execute_sync()` are automatically visible inside callbacks.  This pattern
is useful for per-cell tracing, request IDs, or streaming writers.

In [ ]:
cell_events: contextvars.ContextVar[list] = contextvars.ContextVar("cell_events")

# Attach a collector to the current cell's context
log: list = []
cell_events.set(log)

async def traced_fetch(ctx: BuiltinContext) -> str:
    key = ctx.argv[0] if ctx.argv else "?"
    # Write to the collector without any explicit reference to `log`
    cell_events.get().append(f"fetch:{key}")
    await asyncio.sleep(0)
    return f"result:{key}\n"

bash = Bash(custom_builtins={"fetch": traced_fetch})
result = await bash.execute("fetch alpha; fetch beta; fetch gamma")

print(result.stdout)
print("Captured events:", log)

## ScriptedTool — multi-tool orchestration

`ScriptedTool` lets you register several Python callbacks as distinct bash
commands and compose them in a single script.  The same Jupyter compatibility
guarantees apply.

In [ ]:
def parse_csv(params, stdin=None):
    """Convert CSV stdin to JSON array."""
    lines = (stdin or "").strip().splitlines()
    header, *rows = lines
    cols = header.split(",")
    return json.dumps([dict(zip(cols, r.split(","))) for r in rows]) + "\n"

async def format_report(params, stdin=None):
    """Turn JSON array into a markdown table."""
    records = json.loads(stdin or "[]")
    if not records:
        return "no data\n"
    headers = list(records[0].keys())
    sep = " | "
    header_row = sep.join(headers)
    divider = sep.join("-" * len(h) for h in headers)
    rows = [sep.join(r.get(h, "") for h in headers) for r in records]
    await asyncio.sleep(0)
    return "\n".join([header_row, divider, *rows]) + "\n"

tool = ScriptedTool("report-builder")
tool.add_tool("parse-csv", "Parse CSV to JSON", callback=parse_csv)
tool.add_tool("format-report", "Format JSON as markdown table", callback=format_report)

result = await tool.execute("""
    printf 'name,score,grade\nalice,95,A\nbob,87,B\ncarol,92,A\n' \
      | parse-csv \
      | format-report
""")
print(result.stdout)